## PACOTES ##

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import json
import warnings

from pathlib import Path
from scipy.stats import chi2
from IPython.display import display

In [ ]:

BASE_PATH = Path("thoracic_surgery_base_modelo_statsmodels.csv")
OUTPUT_DIR = Path("outputs")

CAMINHO_MODELO_COMPLETO = OUTPUT_DIR / "modelo_completo.pkl"
CAMINHO_VARIAVEIS_MODELO = OUTPUT_DIR / "variaveis_modelo.json"

CAMINHO_MODELO_FINAL = OUTPUT_DIR / "modelo_final.pkl"
CAMINHO_VARIAVEIS_FINAIS = OUTPUT_DIR / "variaveis_finais.json"
CAMINHO_HISTORICO_SELECAO = OUTPUT_DIR / "historico_selecao.csv"

VAR_RESPOSTA = "obito_1_ano"

ALPHA = 0.05

In [ ]:

dados = pd.read_csv(BASE_PATH)

with open(CAMINHO_VARIAVEIS_MODELO, "r", encoding="utf-8") as arquivo:
    info_variaveis = json.load(arquivo)

print("Base carregada com sucesso.")
print("Dimensão da base:")
print(dados.shape)

print("\nInformações carregadas do Programa 1:")
print(f"Variável resposta: {info_variaveis['variavel_resposta']}")
print(f"Número de observações: {info_variaveis['n_observacoes']}")
print(f"Número de óbitos: {info_variaveis['n_eventos_obito']}")
print(f"Número de não óbitos: {info_variaveis['n_nao_eventos']}")

print("\nPrimeiras linhas da base:")
display(dados.head())

Base carregada com sucesso.
Dimensão da base:
(470, 26)

Informações carregadas do Programa 1:
Variável resposta: obito_1_ano
Número de observações: 470
Número de óbitos: 70
Número de não óbitos: 400

Primeiras linhas da base:


,obito_1_ano,constante,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]","C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0]","C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]",...,hemoptise_antes_cirurgia,dispneia_antes_cirurgia,tosse_antes_cirurgia,fraqueza_antes_cirurgia,diabetes_mellitus_tipo_2,infarto_miocardio_ate_6_meses,doenca_arterial_periferica,tabagismo,asma,idade
0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,60.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,51.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,59.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.0
4,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,73.0


In [ ]:

y = dados[VAR_RESPOSTA].astype(float)

variaveis_modelo = info_variaveis["variaveis_modelo"]

variaveis_modelo = [v for v in variaveis_modelo if v != VAR_RESPOSTA]

X = dados[variaveis_modelo].copy()

for coluna in X.columns:
    X[coluna] = pd.to_numeric(X[coluna], errors="coerce")

faltantes = X.isna().sum()
faltantes = faltantes[faltantes > 0]

if len(faltantes) > 0:
    print("Variáveis com valores ausentes:")
    display(faltantes.to_frame("n_faltantes"))
    raise ValueError("Existem valores ausentes na matriz X.")

print("Dimensão de y:")
print(y.shape)

print("\nDimensão de X:")
print(X.shape)

print("\nVariáveis carregadas para o processo de seleção:")
display(pd.DataFrame({"variavel": X.columns}))

Dimensão de y:
(470,)

Dimensão de X:
(470, 25)

Variáveis carregadas para o processo de seleção:


,variavel
0,constante
1,"C(diagnostico, Treatment(reference='DGN3'))[T...."
2,"C(diagnostico, Treatment(reference='DGN3'))[T...."
3,"C(diagnostico, Treatment(reference='DGN3'))[T...."
4,"C(diagnostico, Treatment(reference='DGN3'))[T...."
5,"C(diagnostico, Treatment(reference='DGN3'))[T...."
6,"C(diagnostico, Treatment(reference='DGN3'))[T...."
7,"C(estado_desempenho_zubrod, Treatment(referenc..."
8,"C(estado_desempenho_zubrod, Treatment(referenc..."
9,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."


In [ ]:

pd.set_option("display.max_colwidth", None)


nomes_constante = ["const", "constante"]

colunas_constante = [
    coluna for coluna in X.columns
    if coluna in nomes_constante or X[coluna].nunique() == 1
]

print("Colunas constantes identificadas:")
print(colunas_constante)


def identificar_bloco(coluna):

    if coluna in colunas_constante:
        return None

    if coluna.startswith("C("):
        inicio = coluna.find("C(") + 2
        fim = coluna.find(",")
        
        if fim != -1:
            return coluna[inicio:fim].strip()

    return coluna

blocos = {}

for coluna in X.columns:
    nome_bloco = identificar_bloco(coluna)
    
    if nome_bloco is None:
        continue
    
    if nome_bloco not in blocos:
        blocos[nome_bloco] = []
    
    blocos[nome_bloco].append(coluna)

tabela_blocos = pd.DataFrame([
    {
        "bloco": nome_bloco,
        "n_colunas": len(colunas),
        "colunas": ", ".join(colunas)
    }
    for nome_bloco, colunas in blocos.items()
])

display(tabela_blocos)

Colunas constantes identificadas:
['constante']


,bloco,n_colunas,colunas
0,diagnostico,6,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1], C(diagnostico, Treatment(reference='DGN3'))[T.DGN2], C(diagnostico, Treatment(reference='DGN3'))[T.DGN4], C(diagnostico, Treatment(reference='DGN3'))[T.DGN5], C(diagnostico, Treatment(reference='DGN3'))[T.DGN6], C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]"
1,estado_desempenho_zubrod,2,"C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0], C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]"
2,tamanho_tumor_tnm,3,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12], C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC13], C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC14]"
3,capacidade_vital_forcada,1,capacidade_vital_forcada
4,volume_expiratorio_forcado_1s,1,volume_expiratorio_forcado_1s
5,dor_antes_cirurgia,1,dor_antes_cirurgia
6,hemoptise_antes_cirurgia,1,hemoptise_antes_cirurgia
7,dispneia_antes_cirurgia,1,dispneia_antes_cirurgia
8,tosse_antes_cirurgia,1,tosse_antes_cirurgia
9,fraqueza_antes_cirurgia,1,fraqueza_antes_cirurgia


In [ ]:

def ajustar_modelo(y, X):

    modelo = sm.GLM(
        y,
        X,
        family=sm.families.Binomial()
    )
    
    resultado = modelo.fit(maxiter=200)
    
    return resultado


def teste_wald_bloco(resultado, colunas_bloco):

    beta = resultado.params[colunas_bloco].values
    matriz_cov = resultado.cov_params().loc[colunas_bloco, colunas_bloco].values
    
    q = len(colunas_bloco)
    
    try:
        matriz_cov_inv = np.linalg.inv(matriz_cov)
    except np.linalg.LinAlgError:
        matriz_cov_inv = np.linalg.pinv(matriz_cov)
    
    estatistica_wald = float(beta.T @ matriz_cov_inv @ beta)
    p_valor_wald = float(chi2.sf(estatistica_wald, q))
    
    return estatistica_wald, p_valor_wald, q


def teste_deviance_modelo_reduzido(y, X_atual, resultado_atual, colunas_remover):

    X_reduzido = X_atual.drop(columns=colunas_remover)
    
    resultado_reduzido = ajustar_modelo(y, X_reduzido)
    
    delta_deviance = float(resultado_reduzido.deviance - resultado_atual.deviance)
    
    if delta_deviance < 0:
        delta_deviance = 0.0
    
    q = len(colunas_remover)
    p_valor_deviance = float(chi2.sf(delta_deviance, q))
    
    return resultado_reduzido, delta_deviance, p_valor_deviance, q

In [ ]:

X_atual = X.copy()
blocos_ativos = blocos.copy()

historico_selecao = []

iteracao = 1

while True:
    
    print("=" * 80)
    print(f"Iteração {iteracao}")
    print("=" * 80)
    
    resultado_atual = ajustar_modelo(y, X_atual)
    
    print(f"Deviance do modelo atual: {resultado_atual.deviance:.6f}")
    print(f"Número de parâmetros: {len(resultado_atual.params)}")
    print(f"Número de blocos ativos: {len(blocos_ativos)}")
    
    resultados_iteracao = []
    
    for nome_bloco, colunas_bloco in blocos_ativos.items():
        
        colunas_bloco = [col for col in colunas_bloco if col in X_atual.columns]
        
        if len(colunas_bloco) == 0:
            continue
        
        estat_wald, p_wald, gl_wald = teste_wald_bloco(
            resultado_atual,
            colunas_bloco
        )
        
        try:
            resultado_reduzido, delta_dev, p_dev, gl_dev = teste_deviance_modelo_reduzido(
                y,
                X_atual,
                resultado_atual,
                colunas_bloco
            )
            
            erro_ajuste = ""
        
        except Exception as erro:
            delta_dev = np.nan
            p_dev = np.nan
            gl_dev = len(colunas_bloco)
            erro_ajuste = str(erro)
        
        remover = (
            p_wald > ALPHA
            and p_dev > ALPHA
            and erro_ajuste == ""
        )
        
        resultados_iteracao.append({
            "iteracao": iteracao,
            "bloco": nome_bloco,
            "n_parametros_bloco": len(colunas_bloco),
            "estatistica_wald": estat_wald,
            "p_valor_wald": p_wald,
            "delta_deviance": delta_dev,
            "p_valor_deviance": p_dev,
            "remover_pelos_criterios": remover,
            "colunas_do_bloco": ", ".join(colunas_bloco),
            "erro_ajuste": erro_ajuste
        })
    
    tabela_iteracao = pd.DataFrame(resultados_iteracao)
    
    tabela_iteracao = tabela_iteracao.sort_values(
        by=["remover_pelos_criterios", "p_valor_wald", "p_valor_deviance"],
        ascending=[False, False, False]
    ).reset_index(drop=True)
    
    display(tabela_iteracao)
    
    candidatos_remocao = tabela_iteracao[
        tabela_iteracao["remover_pelos_criterios"] == True
    ].copy()
    
    if candidatos_remocao.empty:
        print("\nNenhum bloco atende simultaneamente aos dois critérios de remoção.")
        print("Processo de seleção finalizado.")
        break
    
    bloco_removido = candidatos_remocao.iloc[0]["bloco"]
    colunas_removidas = blocos_ativos[bloco_removido]
    
    print("\nBloco removido nesta iteração:")
    print(bloco_removido)
    
    print("\nColunas removidas:")
    for coluna in colunas_removidas:
        print(f"- {coluna}")
    

    historico_selecao.append({
        "iteracao": iteracao,
        "bloco_removido": bloco_removido,
        "n_parametros_removidos": len(colunas_removidas),
        "p_valor_wald": float(candidatos_remocao.iloc[0]["p_valor_wald"]),
        "p_valor_deviance": float(candidatos_remocao.iloc[0]["p_valor_deviance"]),
        "delta_deviance": float(candidatos_remocao.iloc[0]["delta_deviance"]),
        "deviance_modelo_antes": float(resultado_atual.deviance),
        "colunas_removidas": ", ".join(colunas_removidas)
    })
    

    X_atual = X_atual.drop(columns=colunas_removidas)
    del blocos_ativos[bloco_removido]
    
    iteracao += 1

Iteração 1
Deviance do modelo atual: 341.187221
Número de parâmetros: 25
Número de blocos ativos: 16


,iteracao,bloco,n_parametros_bloco,estatistica_wald,p_valor_wald,delta_deviance,p_valor_deviance,remover_pelos_criterios,colunas_do_bloco,erro_ajuste
0,1,asma,1,3.656536e-07,0.999518,0.303253,0.581850,True,asma,
1,1,infarto_miocardio_ate_6_meses,1,3.867849e-07,0.999504,0.563121,0.453005,True,infarto_miocardio_ate_6_meses,
2,1,doenca_arterial_periferica,1,9.520113e-03,0.922273,0.009629,0.921833,True,doenca_arterial_periferica,
3,1,estado_desempenho_zubrod,2,8.073909e-01,0.667847,0.808683,0.667416,True,"C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0], C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]",
4,1,hemoptise_antes_cirurgia,1,2.006611e-01,0.654188,0.197459,0.656780,True,hemoptise_antes_cirurgia,
5,1,idade,1,2.758390e-01,0.599442,0.275079,0.599945,True,idade,
6,1,tosse_antes_cirurgia,1,1.429446e+00,0.231855,1.481631,0.223519,True,tosse_antes_cirurgia,
7,1,capacidade_vital_forcada,1,1.510292e+00,0.219094,1.544362,0.213970,True,capacidade_vital_forcada,
8,1,dor_antes_cirurgia,1,1.657924e+00,0.197884,1.576328,0.209290,True,dor_antes_cirurgia,
9,1,fraqueza_antes_cirurgia,1,1.694964e+00,0.192948,1.637602,0.200655,True,fraqueza_antes_cirurgia,



Bloco removido nesta iteração:
asma

Colunas removidas:
- asma
Iteração 2
Deviance do modelo atual: 341.490473
Número de parâmetros: 24
Número de blocos ativos: 15


,iteracao,bloco,n_parametros_bloco,estatistica_wald,p_valor_wald,delta_deviance,p_valor_deviance,remover_pelos_criterios,colunas_do_bloco,erro_ajuste
0,2,infarto_miocardio_ate_6_meses,1,3.865668e-07,0.999504,0.561148,0.453798,True,infarto_miocardio_ate_6_meses,
1,2,doenca_arterial_periferica,1,9.520337e-03,0.922272,0.009629,0.921832,True,doenca_arterial_periferica,
2,2,estado_desempenho_zubrod,2,7.824627e-01,0.676224,0.783687,0.675810,True,"C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0], C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]",
3,2,hemoptise_antes_cirurgia,1,2.067504e-01,0.649326,0.203400,0.651990,True,hemoptise_antes_cirurgia,
4,2,idade,1,2.693516e-01,0.603767,0.268639,0.604246,True,idade,
5,2,tosse_antes_cirurgia,1,1.447790e+00,0.228883,1.501109,0.220501,True,tosse_antes_cirurgia,
6,2,capacidade_vital_forcada,1,1.467239e+00,0.225782,1.500112,0.220654,True,capacidade_vital_forcada,
7,2,dor_antes_cirurgia,1,1.663892e+00,0.197079,1.581950,0.208480,True,dor_antes_cirurgia,
8,2,fraqueza_antes_cirurgia,1,1.702218e+00,0.191998,1.644456,0.199715,True,fraqueza_antes_cirurgia,
9,2,volume_expiratorio_forcado_1s,1,2.865810e+00,0.090480,4.010478,0.045218,False,volume_expiratorio_forcado_1s,



Bloco removido nesta iteração:
infarto_miocardio_ate_6_meses

Colunas removidas:
- infarto_miocardio_ate_6_meses
Iteração 3
Deviance do modelo atual: 342.051622
Número de parâmetros: 23
Número de blocos ativos: 14


...


Bloco removido nesta iteração:
volume_expiratorio_forcado_1s

Colunas removidas:
- volume_expiratorio_forcado_1s
Iteração 11
Deviance do modelo atual: 349.387970
Número de parâmetros: 14
Número de blocos ativos: 6


,iteracao,bloco,n_parametros_bloco,estatistica_wald,p_valor_wald,delta_deviance,p_valor_deviance,remover_pelos_criterios,colunas_do_bloco,erro_ajuste
0,11,fraqueza_antes_cirurgia,1,2.626412,0.105099,2.506314,0.113391,True,fraqueza_antes_cirurgia,
1,11,diabetes_mellitus_tipo_2,1,5.390816,0.020243,4.874221,0.027261,False,diabetes_mellitus_tipo_2,
2,11,tabagismo,1,5.492146,0.019102,6.810243,0.009064,False,tabagismo,
3,11,tamanho_tumor_tnm,3,11.382932,0.009826,10.843391,0.012604,False,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12], C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC13], C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC14]",
4,11,diagnostico,6,17.013894,0.009232,16.500192,0.011307,False,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1], C(diagnostico, Treatment(reference='DGN3'))[T.DGN2], C(diagnostico, Treatment(reference='DGN3'))[T.DGN4], C(diagnostico, Treatment(reference='DGN3'))[T.DGN5], C(diagnostico, Treatment(reference='DGN3'))[T.DGN6], C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]",
5,11,dispneia_antes_cirurgia,1,6.916107,0.008542,6.147015,0.013163,False,dispneia_antes_cirurgia,



Bloco removido nesta iteração:
fraqueza_antes_cirurgia

Colunas removidas:
- fraqueza_antes_cirurgia
Iteração 12
Deviance do modelo atual: 351.894284
Número de parâmetros: 13
Número de blocos ativos: 5


,iteracao,bloco,n_parametros_bloco,estatistica_wald,p_valor_wald,delta_deviance,p_valor_deviance,remover_pelos_criterios,colunas_do_bloco,erro_ajuste
0,12,diabetes_mellitus_tipo_2,1,5.748853,0.016499,5.168041,0.023006,False,diabetes_mellitus_tipo_2,
1,12,tabagismo,1,6.134679,0.013256,7.747101,0.005380,False,tabagismo,
2,12,tamanho_tumor_tnm,3,10.961826,0.011934,10.327606,0.015977,False,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12], C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC13], C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC14]",
3,12,dispneia_antes_cirurgia,1,6.323880,0.011912,5.640174,0.017553,False,dispneia_antes_cirurgia,
4,12,diagnostico,6,17.319505,0.008178,16.889593,0.009698,False,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1], C(diagnostico, Treatment(reference='DGN3'))[T.DGN2], C(diagnostico, Treatment(reference='DGN3'))[T.DGN4], C(diagnostico, Treatment(reference='DGN3'))[T.DGN5], C(diagnostico, Treatment(reference='DGN3'))[T.DGN6], C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]",



Nenhum bloco atende simultaneamente aos dois critérios de remoção.
Processo de seleção finalizado.


In [ ]:

if len(historico_selecao) > 0:
    tabela_historico = pd.DataFrame(historico_selecao)
else:
    tabela_historico = pd.DataFrame(columns=[
        "iteracao",
        "bloco_removido",
        "n_parametros_removidos",
        "p_valor_wald",
        "p_valor_deviance",
        "delta_deviance",
        "deviance_modelo_antes",
        "colunas_removidas"
    ])

tabela_historico.to_csv(CAMINHO_HISTORICO_SELECAO, index=False, encoding="utf-8-sig")

print("Histórico da seleção salvo em:")
print(CAMINHO_HISTORICO_SELECAO)

print("\nHistórico da seleção:")
display(tabela_historico)

Histórico da seleção salvo em:
outputs\historico_selecao.csv

Histórico da seleção:


,iteracao,bloco_removido,n_parametros_removidos,p_valor_wald,p_valor_deviance,delta_deviance,deviance_modelo_antes,colunas_removidas
0,1,asma,1,0.999518,0.581850,0.303253,341.187221,asma
1,2,infarto_miocardio_ate_6_meses,1,0.999504,0.453798,0.561148,341.490473,infarto_miocardio_ate_6_meses
2,3,doenca_arterial_periferica,1,0.924999,0.924588,0.008960,342.051622,doenca_arterial_periferica
3,4,estado_desempenho_zubrod,2,0.664535,0.664118,0.818591,342.060582,"C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0], C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]"
4,5,idade,1,0.685460,0.685771,0.163702,342.879172,idade
5,6,hemoptise_antes_cirurgia,1,0.629165,0.632028,0.229320,343.042875,hemoptise_antes_cirurgia
6,7,tosse_antes_cirurgia,1,0.371506,0.363997,0.824057,343.272195,tosse_antes_cirurgia
7,8,capacidade_vital_forcada,1,0.270288,0.265296,1.240907,344.096251,capacidade_vital_forcada
8,9,dor_antes_cirurgia,1,0.245797,0.259983,1.268849,345.337159,dor_antes_cirurgia
9,10,volume_expiratorio_forcado_1s,1,0.160590,0.095331,2.781962,346.606008,volume_expiratorio_forcado_1s


In [ ]:

resultado_final = ajustar_modelo(y, X_atual)

resultado_final.save(str(CAMINHO_MODELO_FINAL), remove_data=False)

print("Modelo final salvo em:")
print(CAMINHO_MODELO_FINAL)

print("\nResumo do modelo final:")
print(resultado_final.summary())

Modelo final salvo em:
outputs\modelo_final.pkl

Resumo do modelo final:
                 Generalized Linear Model Regression Results                  
Dep. Variable:            obito_1_ano   No. Observations:                  470
Model:                            GLM   Df Residuals:                      457
Model Family:                Binomial   Df Model:                           12
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -175.95
Date:                Fri, 19 Jun 2026   Deviance:                       351.89
Time:                        09:07:14   Pearson chi2:                     437.
No. Iterations:                    20   Pseudo R-squ. (CS):            0.08881
Covariance Type:            nonrobust                                         
                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------

In [ ]:

informacoes_finais = {
    "arquivo_base": str(BASE_PATH),
    "variavel_resposta": VAR_RESPOSTA,
    "alpha": ALPHA,
    "criterio_remocao": "Remover somente se p_valor_wald > alpha e p_valor_deviance > alpha",
    "n_observacoes": int(resultado_final.nobs),
    "n_eventos_obito": int(y.sum()),
    "n_nao_eventos": int((1 - y).sum()),
    "variaveis_modelo_final": X_atual.columns.tolist(),
    "variaveis_explicativas_modelo_final": [
        col for col in X_atual.columns if col not in ["const", "constante"]
    ],
    "blocos_finais": blocos_ativos,
    "colunas_constantes": colunas_constante,
    "n_parametros_modelo_final": int(len(resultado_final.params)),
    "deviance_modelo_final": float(resultado_final.deviance),
    "graus_liberdade_residuo": int(resultado_final.df_resid),
    "log_verossimilhanca": float(resultado_final.llf),
    "blocos_removidos": tabela_historico["bloco_removido"].tolist()
}

with open(CAMINHO_VARIAVEIS_FINAIS, "w", encoding="utf-8") as arquivo:
    json.dump(informacoes_finais, arquivo, ensure_ascii=False, indent=4)

print("Informações das variáveis finais salvas em:")
print(CAMINHO_VARIAVEIS_FINAIS)

Informações das variáveis finais salvas em:
outputs\variaveis_finais.json


In [ ]:

resumo_selecao = pd.DataFrame({
    "medida": [
        "n_variaveis_inicio",
        "n_variaveis_final",
        "n_blocos_inicio",
        "n_blocos_final",
        "n_blocos_removidos",
        "deviance_modelo_completo",
        "deviance_modelo_final",
        "log_verossimilhanca_modelo_final"
    ],
    "valor": [
        len(X.columns),
        len(X_atual.columns),
        len(blocos),
        len(blocos_ativos),
        len(tabela_historico),
        float(info_variaveis["deviance_modelo_completo"]),
        float(resultado_final.deviance),
        float(resultado_final.llf)
    ]
})

display(resumo_selecao)

print("Variáveis que permaneceram no modelo final:")
display(pd.DataFrame({"variavel_final": X_atual.columns}))

,medida,valor
0,n_variaveis_inicio,25.000000
1,n_variaveis_final,13.000000
2,n_blocos_inicio,16.000000
3,n_blocos_final,5.000000
4,n_blocos_removidos,11.000000
5,deviance_modelo_completo,341.187221
6,deviance_modelo_final,351.894284
7,log_verossimilhanca_modelo_final,-175.947142


Variáveis que permaneceram no modelo final:


,variavel_final
0,constante
1,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]"
2,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]"
3,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]"
4,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]"
5,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]"
6,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]"
7,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12]"
8,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC13]"
9,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC14]"


In [ ]:

if len(historico_selecao) > 0:
    tabela_historico = pd.DataFrame(historico_selecao)
else:
    tabela_historico = pd.DataFrame(columns=[
        "iteracao",
        "bloco_removido",
        "n_parametros_removidos",
        "p_valor_wald",
        "p_valor_deviance",
        "delta_deviance",
        "deviance_modelo_antes",
        "colunas_removidas"
    ])

tabela_historico.to_csv(
    CAMINHO_HISTORICO_SELECAO,
    index=False,
    encoding="utf-8-sig"
)

print("Histórico da seleção salvo em:")
print(CAMINHO_HISTORICO_SELECAO)

print("\nHistórico da seleção:")
display(tabela_historico)

Histórico da seleção salvo em:
outputs\historico_selecao.csv

Histórico da seleção:


,iteracao,bloco_removido,n_parametros_removidos,p_valor_wald,p_valor_deviance,delta_deviance,deviance_modelo_antes,colunas_removidas
0,1,asma,1,0.999518,0.581850,0.303253,341.187221,asma
1,2,infarto_miocardio_ate_6_meses,1,0.999504,0.453798,0.561148,341.490473,infarto_miocardio_ate_6_meses
2,3,doenca_arterial_periferica,1,0.924999,0.924588,0.008960,342.051622,doenca_arterial_periferica
3,4,estado_desempenho_zubrod,2,0.664535,0.664118,0.818591,342.060582,"C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0], C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]"
4,5,idade,1,0.685460,0.685771,0.163702,342.879172,idade
5,6,hemoptise_antes_cirurgia,1,0.629165,0.632028,0.229320,343.042875,hemoptise_antes_cirurgia
6,7,tosse_antes_cirurgia,1,0.371506,0.363997,0.824057,343.272195,tosse_antes_cirurgia
7,8,capacidade_vital_forcada,1,0.270288,0.265296,1.240907,344.096251,capacidade_vital_forcada
8,9,dor_antes_cirurgia,1,0.245797,0.259983,1.268849,345.337159,dor_antes_cirurgia
9,10,volume_expiratorio_forcado_1s,1,0.160590,0.095331,2.781962,346.606008,volume_expiratorio_forcado_1s


In [ ]:

resultado_final = ajustar_modelo(y, X_atual)

resultado_final.save(str(CAMINHO_MODELO_FINAL), remove_data=False)

print("Modelo final salvo em:")
print(CAMINHO_MODELO_FINAL)

print("\nResumo do modelo final:")
print(resultado_final.summary())

Modelo final salvo em:
outputs\modelo_final.pkl

Resumo do modelo final:
                 Generalized Linear Model Regression Results                  
Dep. Variable:            obito_1_ano   No. Observations:                  470
Model:                            GLM   Df Residuals:                      457
Model Family:                Binomial   Df Model:                           12
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -175.95
Date:                Fri, 19 Jun 2026   Deviance:                       351.89
Time:                        09:11:28   Pearson chi2:                     437.
No. Iterations:                    20   Pseudo R-squ. (CS):            0.08881
Covariance Type:            nonrobust                                         
                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------

In [ ]:

informacoes_finais = {
    "arquivo_base": str(BASE_PATH),
    "variavel_resposta": VAR_RESPOSTA,
    "alpha": ALPHA,
    "criterio_remocao": "Remover somente se p_valor_wald > alpha e p_valor_deviance > alpha",
    "n_observacoes": int(resultado_final.nobs),
    "n_eventos_obito": int(y.sum()),
    "n_nao_eventos": int((1 - y).sum()),
    "variaveis_modelo_final": X_atual.columns.tolist(),
    "variaveis_explicativas_modelo_final": [
        col for col in X_atual.columns if col not in ["const", "constante"]
    ],
    "blocos_finais": blocos_ativos,
    "colunas_constantes": colunas_constante,
    "n_parametros_modelo_final": int(len(resultado_final.params)),
    "deviance_modelo_final": float(resultado_final.deviance),
    "graus_liberdade_residuo": int(resultado_final.df_resid),
    "log_verossimilhanca": float(resultado_final.llf),
    "blocos_removidos": tabela_historico["bloco_removido"].tolist()
}

with open(CAMINHO_VARIAVEIS_FINAIS, "w", encoding="utf-8") as arquivo:
    json.dump(informacoes_finais, arquivo, ensure_ascii=False, indent=4)

print("Informações das variáveis finais salvas em:")
print(CAMINHO_VARIAVEIS_FINAIS)

Informações das variáveis finais salvas em:
outputs\variaveis_finais.json


In [ ]:

resumo_selecao = pd.DataFrame({
    "medida": [
        "n_variaveis_inicio",
        "n_variaveis_final",
        "n_blocos_inicio",
        "n_blocos_final",
        "n_blocos_removidos",
        "deviance_modelo_completo",
        "deviance_modelo_final",
        "log_verossimilhanca_modelo_final"
    ],
    "valor": [
        len(X.columns),
        len(X_atual.columns),
        len(blocos),
        len(blocos_ativos),
        len(tabela_historico),
        float(info_variaveis["deviance_modelo_completo"]),
        float(resultado_final.deviance),
        float(resultado_final.llf)
    ]
})

display(resumo_selecao)

print("Variáveis que permaneceram no modelo final:")
display(pd.DataFrame({"variavel_final": X_atual.columns}))

,medida,valor
0,n_variaveis_inicio,25.000000
1,n_variaveis_final,13.000000
2,n_blocos_inicio,16.000000
3,n_blocos_final,5.000000
4,n_blocos_removidos,11.000000
5,deviance_modelo_completo,341.187221
6,deviance_modelo_final,351.894284
7,log_verossimilhanca_modelo_final,-175.947142


Variáveis que permaneceram no modelo final:


,variavel_final
0,constante
1,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]"
2,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]"
3,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]"
4,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]"
5,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]"
6,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]"
7,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12]"
8,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC13]"
9,"C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC14]"
